# A free GPU for the song site

Open it with [this link](https://colab.research.google.com/github/soluzka/AI-driven_NVIDIA_song_designer/blob/main/deploy/colab/song_gpu.ipynb),
choose **Runtime → Change runtime type → T4 GPU**, and press **Run all**. Nothing
is installed on your own computer, nothing is rented, and no account or card is
needed.

What each cell does, so a slow one is recognisable rather than a spinner:

| Cell | Prints | Takes |
| --- | --- | --- |
| the card | `torch … cuda True` | seconds |
| packages | `packages in place` | 3–6 minutes |
| weights | download bars, then `weights exit 0` | 5–15 minutes |
| the service | `service: {...}` | 1–3 minutes (the model is loading) |
| the tunnel | `ADDRESS: https://…` | under a minute |

The last cell prints a line beginning `ADDRESS:`. Press **Connect my model** in
the site header and paste it. If you have the owner's join key, put it in the
`JOIN_KEY` cell above that and the card joins the pool instead.

Do not paste this file's text into a cell: a notebook is JSON and Colab runs
pasted text as Python, which is the `NameError: name 'null' is not defined`
people see. The link above opens it as a notebook. If that link will not load,
download `/song_gpu.ipynb` from the site and use **File → Upload notebook**.

The card goes away when the session ends — that is the trade for free. A T4 has
16 GB, twice the card the site was built on, so songs render faster while it runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The song service's own packages. Colab already ships a CUDA build of torch and
# torchaudio, so they are deliberately not named here: a pip wheel from PyPI would
# replace them with a CPU build and quietly move the model off the card.
print('1/4 system packages ...', flush=True)
!apt-get -qq install -y ffmpeg libsndfile1 git > /dev/null
print('2/4 the code ...', flush=True)
!git clone --depth 1 https://github.com/soluzka/AI-driven_NVIDIA_song_designer.git /content/internet 2>/dev/null || true
print('3/4 the packages (a few minutes, no output until pip finishes) ...', flush=True)
!pip install -q flask flask-cors soundfile librosa scipy demucs hf_transfer spacy
!pip install -q transformers==4.57.6 huggingface-hub==0.36.2 numpy==2.4.6
# ACE-Step's metadata pins transformers==4.50.0 against the 4.57.6 the lyric model
# needs, so the model's code is installed without its dependency list.
!pip install -q --no-deps git+https://github.com/ace-step/ACE-Step.git
print('4/4 checking the imports ...', flush=True)
import torch
from acestep.pipeline_ace_step import ACEStepPipeline
print('packages in place; torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The two checkpoints (about 10 GB, into the session disk): ACE-Step v1 3.5B for
# the music, Qwen2.5 1.5B for the words. install.py reads song_service/models.json
# and resumes rather than restarting a partial folder.
import os, subprocess
os.environ['HF_HOME'] = '/content/hf'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('downloading about 10 GB of weights; the bars below are the progress', flush=True)
result = subprocess.run(['python', 'song_service/install.py', '--weights'], cwd='/content/internet')
print('weights exit', result.returncode)
!du -sh /content/internet/song_service/models/* 2>/dev/null || true

In [ ]:
# The service on the notebook's own GPU, on every interface, warming the model now
# so the first visitor does not wait for the load. The wait prints, because a
# silent loop is indistinguishable from a hung one.
import os, subprocess, time, urllib.request
os.environ.update(SONG_PORT='8090', SONG_HOST='0.0.0.0', SONG_WARM='1', HF_HOME='/content/hf')
print('starting the service ...', flush=True)
log = open('/content/song.log', 'w')
proc = subprocess.Popen(['python', 'app.py', '--role', 'song'], cwd='/content/internet/song_service',
                        stdout=log, stderr=subprocess.STDOUT)
health = ''
for attempt in range(36):
    try:
        health = urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=5).read().decode()
        break
    except Exception:
        if attempt and attempt % 6 == 0:
            print('  waiting for /health ... ' + str(attempt * 5) + 's', flush=True)
        time.sleep(5)
if health:
    print('service:', health[:300])
else:
    print('the service did not answer within 3 minutes. Last lines of /content/song.log:')
    print()
    print(open('/content/song.log').read()[-2000:])
    print()
    print('Rerun this cell after reading that: the usual causes are a package that did')
    print('not install in the cell above, or the GPU runtime not being selected.')

In [ ]:
# Optional. Paste the site owner's join key between the quotes to add this card to
# the pool, so every visitor's click can use it. Leave it empty and the next cell
# just prints an address for Connect my model.
#
# This is a cell and not a prompt on purpose: getpass waits for a terminal the
# notebook does not give it, so the cell sat loading forever and no address was
# ever printed.
JOIN_KEY = ''

In [ ]:
# A free tunnel, so the Cloudflare page can reach this card. Nothing is opened on
# the notebook host: cloudflared dials out, exactly like the launcher does at home.
# Pinned version, as on the PC, so a notebook run is reproducible.
import re, time, json, subprocess, urllib.request
JOIN_KEY = globals().get('JOIN_KEY', '').strip()
print('1/2 downloading cloudflared (40 MB, no output while it downloads) ...', flush=True)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print('2/2 starting the tunnel ...', flush=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090',
                           '--logfile', '/content/tunnel.log'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
url = ''
for attempt in range(30):
    try:
        text = open('/content/tunnel.log').read()
        found = re.findall(r'https://[a-z0-9-]+[.]trycloudflare[.]com', text)
        if found:
            url = found[-1]
            break
    except FileNotFoundError:
        pass
    if attempt and attempt % 5 == 0:
        print('  still waiting for a name ... ' + str(attempt * 2) + 's', flush=True)
    time.sleep(2)

print()
print('=' * 70)
print('tunnel:', url or 'no name after 60 seconds')
print('=' * 70)
if url:
    print()
    print('ADDRESS:', url)
    print('(that line is the whole address; it stops working when this session ends)')
    print()
    print('On the site: press Connect my model in the header and paste that address.')
    if JOIN_KEY:
        body = json.dumps({'url': url, 'note': 'colab t4'}).encode()
        request = urllib.request.Request('https://song-pool.bpiersing.workers.dev/join', data=body,
                                         headers={'content-type': 'application/json', 'x-join-key': JOIN_KEY})
        try:
            print('pool:', urllib.request.urlopen(request, timeout=20).read().decode())
            print('This card is in the pool until the notebook stops. Keep the tab open.')
        except Exception as exc:
            print('join refused:', exc)
            print('Use the address above with Connect my model instead.')
    else:
        print('No JOIN_KEY set, so this card is not in the pool: use the address above.')
else:
    print()
    print('cloudflared did not produce a name. Last lines of /content/tunnel.log:')
    try:
        print(open('/content/tunnel.log').read()[-1500:])
    except Exception as exc:
        print('no log at all:', exc)
        print('That means the download above failed; rerun this cell.')